# CH3-1. ExtraTree Training

## Preamble

In [1]:
# Standard Library
import os
from pathlib import Path

# Data and Numerical Processing
import numpy as np
import pandas as pd

# Machine Learning and Modelling
from sklearn.ensemble import VotingRegressor, ExtraTreesRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_friedman1
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# Visualisation
import DataGraph as dg

In [2]:
# Remove inherited Slurm environment from the interactive parent job
for key in list(os.environ):
    if key.startswith(("SLURM_", "SBATCH_", "SRUN_")):
        os.environ.pop(key)

### Directories

In [3]:
# Directory
PROJECT_DIR = Path("/scratch/project_2015384/Hanseul")
BASE_DIR = PROJECT_DIR / "Codes" / "SmartSim"
MAIN_DIR = BASE_DIR / "OnlineInference"
DATASET_DIR = MAIN_DIR / "dataset"
SAVE_DIR = MAIN_DIR / "model"

for directory in [PROJECT_DIR, BASE_DIR, MAIN_DIR, DATASET_DIR, SAVE_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Path
FILE_PATH = DATASET_DIR / "california_housing.csv"

### Configuration

In [4]:
# System configuration
SEED = 42
np.random.seed(SEED)
N_JOBS = len(list(os.sched_getaffinity(0)))

# SmartRedis configuration
REDIS_PORT = 2026

In [5]:
# ONNX configuration (Backend ONNX ver: 1.21.0 -> opset: 26, IR: 13)
OPSET_VERSION = 26
IR_VERSION = 13

In [6]:
# Dataset configuration
N_SAMPLES = 10000
SIGMA_NOISE = [0.2, 0.5, 0.8]  # Noise levels for the Friedman dataset

### Metric

In [7]:
# Metric computation functions (MAE, MSE, R2)
def compute_metrics(model, X_test, y_test, verbose=True):
    # Compute metrics for the given model and test data
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    # Print the computed metrics
    summary = dg.TableMaker(
        title="Model Evaluation Metrics",
        columns=["Metric", "Value"],
    )
    summary.add_row("MAE", f"{mae:.4f}")
    summary.add_row("MSE", f"{mse:.4f}")
    summary.add_row("R2", f"{r2:.4f}")
    
    if verbose:
        summary.display()
    
    return (mae, mse, r2)

### Dataset I/O

In [8]:
# Generate the Friedman dataset with specified noise levels
def generate_friedman_dataset(n_samples, n_features, noise_levels, random_state=SEED):
    datasets = {}
    for noise in noise_levels:
        X, y = make_friedman1(n_samples=n_samples, n_features=n_features, noise=noise, random_state=random_state)
        datasets[noise] = (X, y)
    return datasets

datasets = generate_friedman_dataset(N_SAMPLES, n_features=10, noise_levels=SIGMA_NOISE)

# Combine all datasets into a single DataFrame for analysis
X = np.vstack([datasets[noise][0] for noise in SIGMA_NOISE])
y = np.hstack([datasets[noise][1] for noise in SIGMA_NOISE])

# Split the combined dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

## ML Regression

### Train and Inference

In [9]:
# Define extra trees regressor model
et = ExtraTreesRegressor(n_estimators=100, random_state=SEED, n_jobs=N_JOBS)

# Train the model
et.fit(X_train, y_train)

# Evaluate the model
metrics = compute_metrics(et, X_test, y_test, verbose=True)

Metric,Value
MAE,0.2986
MSE,0.2113
R2,0.9914


In [10]:
node_counts = np.array([
    estimator.tree_.node_count
    for estimator in et.estimators_
])

print({
    "n_estimators": len(node_counts),
    "total_nodes": int(node_counts.sum()),
    "mean_nodes": float(node_counts.mean()),
    "min_nodes": int(node_counts.min()),
    "max_nodes": int(node_counts.max()),
})

{'n_estimators': 100, 'total_nodes': 1982300, 'mean_nodes': 19823.0, 'min_nodes': 19823, 'max_nodes': 19823}


### Convert to ONNX and Save Model

In [11]:
# Define initial input types based on features shape
initial_type = [("float_input", FloatTensorType([None, X_train.shape[1]]))]

# Convert the trained model to ONNX format
onnx_model_et = convert_sklearn(et, initial_types=initial_type, target_opset=OPSET_VERSION)

# Save the ONNX model
onnx_model_paths = {
    "extra_trees_regressor": SAVE_DIR / "extra_trees_regressor.onnx",
}

model_obj = onnx_model_et
path = onnx_model_paths["extra_trees_regressor"]
with open(path, "wb") as f:
    f.write(model_obj.SerializeToString())

/ROIHU_TYKKY_u2ZGue8/miniforge/envs/env1/lib/python3.12/site-packages/skl2onnx/common/_topology.py:1658: UserWarning: Parameter target_opset 26 > 22 is higher than the the latest tested version.
  warnings.warn(
